# D-MTHD implicit-abuse benchmark

Right panel: Accelerator **GPU T4 x2**, Internet **On**. No dataset needs to be attached: both
corpora are downloaded from the Hugging Face Hub by the first cell of the run.

This is the benchmark the paper's goal actually needs. The tweet corpus hides abuse-by-implication
inside a catch-all class and the Wikipedia corpus is a binary attack flag, so on neither of them does
a student ever see an example labelled as implicit. Here the label **is** the distinction:
`not_hate`, `explicit_hate`, `implicit_hate`, built from ISHate and Implicit Hate Corpus stage 1.

Two things to know about the corpus before reading any result from it.

* Every sarcasm-probe text is deleted from all three splits, so probe numbers stay measured on text
  no model has trained on. That costs 1,581 rows and keeps every previously reported probe number
  comparable.
* The classical TF-IDF floor scores 0.747 F1 on explicit hate and 0.453 on implicit hate. That
  29-point gap inside one corpus is the headroom this benchmark exists to measure.

**This is a multi-version run.** Kaggle stops a session at 12 hours. Each version does as much as it
can, then stops itself at 11 hours so the packaging cell below still runs. To continue: add this
version's output as an input (Add Input -> Your Work) and Save & Run All again. The resume path is
detected automatically and finished work is skipped.

This notebook measures the benchmark itself: how well each student reads implication when
implication is what it is trained on. The implicit *specialist teacher* is a different thing and is
built inside the tweet and Wikipedia runs, where it has a committee to join; here it would only
duplicate the task teacher, so `SPECIALIST=0`.

Download `dmthd_implicit_results.tgz` from the output: metrics, histories, predictions and tables,
without the model weights.


In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])          # works if the repo is public
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        z = glob.glob("/kaggle/input/**/dmthd-p3-code.zip", recursive=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        if z:                                                            # zip attached as-is
            zipfile.ZipFile(z[0]).extractall(DEST)
        elif tree:                                                       # Kaggle auto-extracted the zip
            root = os.path.dirname(os.path.dirname(os.path.dirname(tree[0])))
            shutil.copytree(root, DEST)
        else:
            raise SystemExit("clone failed and no code found among the inputs: attach the code dataset")
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print(open("README.md").read()[:400])

In [ ]:
import os, subprocess, glob
env = dict(os.environ, ROOT='/kaggle/working', PYTHONPATH='src', GPU='1', SEEDS='1,2,3',
           COMMITTEES='homo,hetero', MODES='ft,skd,uniform,dmthd',
           TEACHERS='bert-large-uncased:bert-large,GroNLP/hateBERT:hatebert,cardiffnlp/twitter-roberta-base-irony:irony',
           STUDENTS='google/bert_uncased_L-4_H-256_A-4:bert-mini,google/bert_uncased_L-4_H-512_A-8:bert-small,distilbert-base-uncased:distilbert')
# SPECIALIST is off here: on this benchmark the implicit specialist would be a second copy of the
# task teacher. It is trained from these runs and used on the tweet and Wikipedia benchmarks.
env['SPECIALIST'] = '0'
env['TIME_BUDGET_S'] = '39600'   # stop launching work after 11 h so the packaging cell below still runs
env['ABLATION_SEEDS'] = '1'      # ablations are supporting evidence: one seed, stated in Limitations
env['IMPLICIT_RAW'] = '/kaggle/working/raw'
cands = sorted({p.replace(chr(92), '/').split('/runs/')[0] for p in glob.glob('/kaggle/input/**/runs/implicit', recursive=True)})
env['RESUME_FROM'] = ','.join(cands)   # every attached previous output is merged, richest last
print('resume sources:', cands or '(none: starting fresh)', flush=True)
cmd = ['python', 'kaggle/run_benchmark.py', '--dataset', 'implicit', '--stage', 'all']
print('running:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise SystemExit(f'BENCHMARK FAILED with exit code {r.returncode}: scroll up in this log to the first Traceback')
print('BENCHMARK FINISHED')


In [ ]:
# Package a results-only archive plus the specialist checkpoint. The specialist is the one set of
# weights worth downloading from this notebook: the tweet and Wikipedia runs need it as a teacher
# source, and re-training it there would waste the same GPU-hours twice.
import glob, os, subprocess, tarfile
ROOT, DS = '/kaggle/working', 'implicit'
keep = []
for pat in ('results.json', 'eval_*.json', 'history.csv', 'test_probs.npy', 'test_labels.npy',
            'quantize_eval.json', 'transfer_*.json', 'dropped_teachers.json', 'implicit_analysis.json'):
    keep += glob.glob(f'{ROOT}/runs/{DS}/**/{pat}', recursive=True)
keep += glob.glob(f'{ROOT}/runs/{DS}/*.csv') + glob.glob(f'{ROOT}/runs/{DS}/weight_routing/*')
keep += glob.glob(f'{ROOT}/cache/{DS}/meta.json') + glob.glob(f'{ROOT}/cache/{DS}/tau_diagnostic.csv')
keep += glob.glob(f'{ROOT}/data/{DS}/report.json')
subprocess.run(['python', '-m', 'dmthd.tables', '--runs', f'{ROOT}/runs/{DS}', '--data', f'{ROOT}/data/{DS}',
                '--out', f'{ROOT}/tables_{DS}'], env=dict(os.environ, PYTHONPATH='src'))
keep += glob.glob(f'{ROOT}/tables_{DS}/*')
out = f'{ROOT}/dmthd_{DS}_results.tgz'
with tarfile.open(out, 'w:gz') as t:
    for f in keep:
        t.add(f, arcname=os.path.relpath(f, ROOT))
print(f'{len(keep)} files -> {out} ({os.path.getsize(out) / 2**20:.1f} MB) : download this one')

spec = f'{ROOT}/runs/implicit/specialist'
if os.path.isdir(spec):
    sout = f'{ROOT}/dmthd_implicit_specialist.tgz'
    with tarfile.open(sout, 'w:gz') as t:
        t.add(spec, arcname='runs/implicit/specialist')
    print(f'specialist weights -> {sout} ({os.path.getsize(sout) / 2**20:.1f} MB) : '
          f'attach this notebook output to the tweet run so its committee can use it')
else:
    print('no specialist checkpoint here (expected: SPECIALIST=0 on this benchmark). '
          'The tweet run trains it from this corpus on its own.')
print('finished runs:', len(glob.glob(f'{ROOT}/runs/{DS}/*/*/seed*/results.json')))
